# Ensemble Learning and Random Forests

When a complex question is asked to a large number of random people, and the answer is aggregated, the results are usualy better than an expert's answer. This is called _wisdom of the crowd_. Similarly, if we aggregate the predictions of a group of predictors, we will make better predictions than a the best individual predictors. A group of predictors is alled an _ensemble_, and this technique is known as Ensemble Learning, and an Ensemble Learning Algorithm is called an Ensemble method. 

An ensemble of Decision Trees is known as _Random Forest_. These are one of the most powerful ML algorithms available today, despite it's simplicity.

## Voting Classifiers

Suppose we have multiple types of classifiers with ~80% accuracy in each. A simple way to create a better classifier is to aggregate the predictions of each classifier and predict the class that gets the most votes. This is called **hard voting classifier**. 

This voting classifier often achieves a higher accuracy that the best classifier in the ensemble. In fact, even if each classifier is a weak learner (only slightly better than guessing), the ensemble can still be a strong learner, provided there are a sufficient number of weak learners and they are sufficiently diverse. 

This can be explained with something called as _law of large numbers_. If a biased coin (51% heads, 49% tails) is tossed a large number of times, the probablity of obtaining majority of heads keeps on increasing. So if an ensemble has 1000 classifiers which have a 51% accuracy (or any number just better than random), the voting classifier can have upto 75% accuracy.

**Note**: Ensemble methods work best when predictors are as independent from one another as possible. 

In [1]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard')
voting_clf.fit(X_train, y_train)

,estimators,"[('lr', ...), ('rf', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [3]:
from sklearn.metrics import accuracy_score

for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.9
RandomForestClassifier 1.0
SVC 1.0
VotingClassifier 1.0


### Soft Voting

If all classifiers are able to estimate class probablities (with `predict_proba()` method), then Scikit-Learn can predict the class with the highest class probablity, averaged out over all individual classifiers. This is called _soft voting_. 

It often achieves higher performance than hard voting because it gives more weight to highly confident votes. 

## Bagging and Pasting

- When sampling is performed with replacement, this method is called _bagging_ (short for Bootstrap AGGregating).
- When sampling is performed without replacement, it called _pasting_.
- Bagging and pasting involves training several predictors on different random samples of the training set.
- The ensemble can make a prediction for a new instance by simply aggregating the predictions of all predictors. The aggregation is typically the statistical mode for classification, and the average for regression.
- Aggregation reduces both bias and variance. 

In [4]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=min(100, X_train.shape[0]), bootstrap=True, n_jobs=-1)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

In [5]:
accuracy_score(y_test, y_pred)

0.9

## Out-of-Bag Evaluation

- During bagging, some instances may be sampled several times for any given predictor, while others may not be sampled at all.
- By default a BaggingClassifier samples `m` training instances with replacement (`bootstrap=True`), where m is the size of the training set. This means that only about 63% of the training instances are sampled on average for each predictor.6 The remaining 37% of the training instances that are not sampled are called out-of-bag (oob) instances.
- Since a predictor never sees the oob instances during training, it can be evaluated on these instances, without need for a separate validation set. 

In [6]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500, 
    bootstrap=True, n_jobs=-1, oob_score=True)

bag_clf.fit(X_train, y_train)

,estimator,DecisionTreeClassifier()
,n_estimators,500
,max_samples,1.0
,max_features,1.0
,bootstrap,True
,bootstrap_features,False
,oob_score,True
,warm_start,False
,n_jobs,-1
,random_state,None
,verbose,0


In [7]:
bag_clf.oob_score_

0.9125

In [8]:
from sklearn.metrics import accuracy_score

y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.95

In [9]:
bag_clf.oob_decision_function_

array([[0.5978836 , 0.4021164 ],
       [0.01639344, 0.98360656],
       [0.        , 1.        ],
       [1.        , 0.        ],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.29050279, 0.70949721],
       [0.        , 1.        ],
       [0.17073171, 0.82926829],
       [0.94240838, 0.05759162],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.84126984, 0.15873016],
       [0.66321244, 0.33678756],
       [0.07692308, 0.92307692],
       [0.        , 1.        ],
       [0.6627907 , 0.3372093 ],
       [0.8956044 , 0.1043956 ],
       [0.78421053, 0.21578947],
       [1.        , 0.        ],
       [0.0297619 , 0.9702381 ],
       [0.93678161, 0.06321839],
       [1.        , 0.        ],
       [0.09359606, 0.90640394],
       [0.56218905, 0.43781095],
       [0.01754386, 0.98245614],
       [0.        , 1.        ],
       [0.44502618, 0.55497382],
       [0.        , 1.        ],
       [0.18686869, 0.81313131],
       [0.

## Random Patches and Random Subspaces

- `BaggingClassifier` supports sampling the features as well. Sampling is controlled by two hyperparameters: `max_features` and `bootstrap_features`. They work the same way as `max_samples` and `bootstrap` but for feature sampling instead of instance sampling. Thus, each predictor will be trained on a random subset of the input features.
- This technique is useful with high-dimensional inputs (like images).
- Sampling both training instances and features is called the _Random Patches method_.
- Keeping all training instances (`bootstrap=False`; `max_samples=1.0`) but sampling features (`bootstrap_features=True`; `max_features<1.0` is called _Random Subspaces method_.  

## Random Forests

_Random Forest_ is an ensemble of Decision Trees, generally trained via the bagging method, typically with `max_samples` set to the size of the training set.

In [10]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(X_train, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,16
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
y_pred_rf = rnd_clf.predict(X_test)

In [12]:
accuracy_score(y_test, y_pred_rf)

1.0

### Extra-Trees

- It is possible to make trees in a Random Forest even more random by using random thresholds for each feature rather than searching for the best possible thresholds.
- A forest of such extremely random trees is called an _Extremely Randomized Trees ensemble_ (Extra-Trees for short).
- This techniques trades more bias for lower variance. It also makes Extra-Trees much faster to train than regular Random Forests, because finding the best possible threshold for each feature at every node is one of the most time-consuming tasks of growing a tree.
- Can be done in Scikit-Learn using `ExtraTreesClassifier` and `ExtraTreesRegressor`. 

In [13]:
from sklearn.ensemble import ExtraTreesClassifier

etr_clf = ExtraTreesClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
etr_clf.fit(X_train, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,16
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [14]:
y_pred_etr = etr_clf.predict(X_test)

In [15]:
accuracy_score(y_test, y_pred_etr)

1.0

### Feature Importance

- Random Forests make it easy to measure the relative importance of each feature.
- Scikit-Learn measures a feature's imporance by looking at how much the tree nodes that use that feature reduce impurity on average.
- It is a weighted average, where each node's weight is equal to the number of training samples that are associated with it.

In [16]:
from sklearn.datasets import load_iris

iris = load_iris()
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris["data"], iris["target"])
for name, score in zip(iris["feature_names"], rnd_clf.feature_importances_):
    print(name, score)

sepal length (cm) 0.09076735327777141
sepal width (cm) 0.025180008132066365
petal length (cm) 0.418766900409159
petal width (cm) 0.4652857381810032


## Boosting

Boosting (hypothesis boosting) refers to any Ensemble method that can combine several weak learners into a strong learner. The general idea of boosting is to train predictors sequentially, each trying to correct its predecessor.

### AdaBoost (Adaptive Boosting)

- One way for a new predictor to correct its predecessor is to pay a bit more attention to the training instances that the predecessor underfitted.
- This results in new predictors focusing more and more on the hard cases. This is the technique  used by _AdaBoost_.
- For example, when training an AdaBoost classifier, the algorithm first trains a base classifier and uses it to make predictions on the training set. The algorithm decreases the weight of misclassified training instances. Then it trains a second classifier on the updated weights, and then the process repeats.
- Once all predictors are trained, the ensemble makes predictions very much like bagging or pasting, except that predictors have weights assigned depending on their accuracy on the weighted training set.

**Note**: An important drawback of AdaBoost is that it cannot be parallelized (or only partially), since one predictor can only be trained when the previous  one is trained and evaluated. 

- Scikit-Learn uses multiclass version of AdaBoost called (SAMME; Stagewise Additive Modeling using a Multiclass Exponential loss function).
- With two classes, SAMME is equivalent to AdaBoost.
- If predictors can estimate class probablitites (with `predict_proba()`) then Scikit-Learn uses a variant called SAMME.R (R for real) which relies on class probablities rather than predictions and generally performs better. 

In [17]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200, learning_rate=0.5)
ada_clf.fit(X_train, y_train)

,estimator,DecisionTreeC...r(max_depth=1)
,n_estimators,200
,learning_rate,0.5
,algorithm,'deprecated'
,random_state,None
,criterion,'gini'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [18]:
y_pred = ada_clf.predict(X_test)
accuracy_score(y_test, y_pred)

1.0

### Gradient Boosting

- Like AdaBoost, this also works by sequentially adding predictors to an ensemble each one correcting its predeccesor.
- Instead of tweaking the insance weights at every iteration like AdaBoost does, this method tries to fit the new predictor to the residual errors made by the previous predictor.
- **Gradient Boosted Regression Trees (GBRT)**: Gradient boosting with Decision Trees as base predictors. 

In [19]:
from sklearn.tree import DecisionTreeRegressor

tree_reg1 = DecisionTreeRegressor(max_depth=2)
tree_reg1.fit(X, y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,2
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [20]:
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2)
tree_reg2.fit(X, y2)

,criterion,'squared_error'
,splitter,'best'
,max_depth,2
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [21]:
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2)
tree_reg3.fit(X, y3)

,criterion,'squared_error'
,splitter,'best'
,max_depth,2
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [23]:
y_pred = sum(tree.predict(X_test) for tree in (tree_reg1, tree_reg2, tree_reg3))

In [27]:
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0)
gbrt.fit(X, y)

,loss,'squared_error'
,learning_rate,1.0
,n_estimators,3
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,2
,min_impurity_decrease,0.0
,init,None


#### Shrinkage

- Regularization technique where we put `learning_rate` to a low value (such as 0.1) so we need more trees in the ensemble to fit the training set, but it generalizes better.
- To find the optimal number of trees, we can use early stopping.

In [29]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_val, y_train, y_val = train_test_split(X, y)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120)
gbrt.fit(X_train, y_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,120
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,2
,min_impurity_decrease,0.0
,init,None


In [30]:
errors = [mean_squared_error(y_val, y_pred) for y_pred in gbrt.staged_predict(X_val)] # early stopping
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators)
gbrt_best.fit(X_train, y_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,np.int64(120)
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,2
,min_impurity_decrease,0.0
,init,None


It is also possible to implement early stopping by actually stopping early (instead of training large number of trees and finding optimal number) by setting `warm_start=True`.

In [32]:
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True)
min_val_error = float("inf")
error_going_up = 0
for n_estimators in range(1, 120):
    gbrt.n_estimators = n_estimators
    gbrt.fit(X_train, y_train)
    y_pred = gbrt.predict(X_val)
    val_error = mean_squared_error(y_val, y_pred)
    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 5:
            break # early stopping

GradientBoostingRegressor has a `subsample` hyperparameter, which specifies the fraction of training instances to be used for training each tree (0.25 -> each tree trained on randomly selected 25% of instances). Trades higher bias for a lower variance. It also speeds up training. This is called _Stochastic Gradient Boosting_. 

#### XGBoost

- Stands for Extreme Gradient Boosting.
- Designed to be extremely fast, scalable, and portable. 

In [34]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 5.8 MB/s  0:00:23 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 6.4 MB/s  0:00:46 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]


In [35]:
import xgboost

xgb_reg = xgboost.XGBRegressor()
xgb_reg.fit(X_train, y_train)
y_pred = xgb_reg.predict(X_val)